# 🎨 ArtDapter Training on Google Colab

Notebook điều phối training ArtDapter trên Colab Free (T4 16GB).

**Lưu ý:**
- Colab Free session tối đa ~12h → cần checkpoint & resume
- Checkpoint lưu trên Google Drive để không mất khi session reset
- Effective batch size = `batch_size × accumulate_grad_batches` = 4 × 6 = 24

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi
import torch
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"\n✅ GPU: {gpu_name}")
print(f"✅ VRAM: {vram_gb:.1f} GB")

if vram_gb < 15:
    print("⚠️ VRAM < 15GB — có thể cần giảm batch_size xuống 2 trong config")
elif vram_gb >= 35:
    print("💡 GPU lớn — có thể tăng batch_size lên 8-12 trong config để train nhanh hơn")

## 2. Mount Google Drive & Clone Repo

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Tạo thư mục checkpoint trên Drive
!mkdir -p /content/drive/MyDrive/ArtDapter/ckpt/trained
!mkdir -p /content/drive/MyDrive/ArtDapter/ckpt/init

In [ ]:
# === SỬA URL GITHUB CỦA BẠN Ở ĐÂY ===
GITHUB_REPO = "https://github.com/YOUR_USERNAME/ArtDapter.git"

!git clone {GITHUB_REPO} /content/ArtDapter
%cd /content/ArtDapter

## 3. Cài Dependencies

In [ ]:
!pip install -q pytorch-lightning lightning transformers diffusers \
    omegaconf einops wandb datasets safetensors open-clip-torch tqdm Pillow

## 4. Tải Pre-trained Weights

Chỉ cần chạy cell này **1 lần đầu tiên**. Các lần sau weights đã có trên Drive.

In [ ]:
import os

DRIVE_INIT = '/content/drive/MyDrive/ArtDapter/ckpt/init'
LOCAL_INIT = '/content/ArtDapter/ckpt/init'

# --- SD v1.5 weights ---
sd_path = f'{DRIVE_INIT}/v1-5-pruned.ckpt'
if not os.path.exists(sd_path):
    print('📥 Downloading Stable Diffusion v1.5 weights (~4GB)...')
    !wget -q --show-progress -O {sd_path} \
        https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5/resolve/main/v1-5-pruned.ckpt
else:
    print('✅ SD v1.5 weights already on Drive')

# --- ELLA weights ---
ella_path = f'{DRIVE_INIT}/ella-sd1.5-tsc-t5xl.safetensors'
if not os.path.exists(ella_path):
    print('📥 Downloading ELLA weights (~200MB)...')
    !wget -q --show-progress -O {ella_path} \
        https://huggingface.co/QQGYLab/ELLA/resolve/main/ella-sd1.5-tsc-t5xl.safetensors
else:
    print('✅ ELLA weights already on Drive')

# Symlink Drive weights vào local repo
!ln -sf {DRIVE_INIT}/v1-5-pruned.ckpt {LOCAL_INIT}/v1-5-pruned.ckpt
!ln -sf {DRIVE_INIT}/ella-sd1.5-tsc-t5xl.safetensors {LOCAL_INIT}/ella-sd1.5-tsc-t5xl.safetensors
print('\n✅ Weights linked to local repo')

## 5. Prepare Initial Weights

Merge SD v1.5 weights + random ArtDapter weights → `init.ckpt`. Chỉ cần chạy **1 lần**.

In [ ]:
init_ckpt = f'{DRIVE_INIT}/init.ckpt'

if not os.path.exists(init_ckpt):
    print('🔧 Preparing initial weights...')
    !python prepare_weights.py \
        --init_dir {LOCAL_INIT} \
        --output init.ckpt \
        --config configs/train_config_colab.yaml
    # Copy lên Drive
    !cp {LOCAL_INIT}/init.ckpt {init_ckpt}
    print('✅ Saved init.ckpt to Drive')
else:
    # Symlink từ Drive
    !ln -sf {init_ckpt} {LOCAL_INIT}/init.ckpt
    print('✅ init.ckpt already on Drive, linked to local')

## 6. Login WandB

In [ ]:
import wandb
wandb.login()

## 7. 🚀 Start / Resume Training

- **Lần đầu**: chạy cell bên dưới (không có `--resume_from`)
- **Resume**: chạy cell tìm checkpoint trước, rồi chạy cell train với `--resume_from`

In [ ]:
# Tìm checkpoint mới nhất trên Drive (nếu có)
import glob

CKPT_DIR = '/content/drive/MyDrive/ArtDapter/ckpt/trained'
ckpts = sorted(glob.glob(f'{CKPT_DIR}/*.ckpt'))

# Loại bỏ EXCEPTION checkpoints, ưu tiên normal checkpoints
normal_ckpts = [c for c in ckpts if 'EXCEPTION' not in c]
exception_ckpts = [c for c in ckpts if 'EXCEPTION' in c]

if normal_ckpts:
    RESUME_CKPT = normal_ckpts[-1]
    print(f'📌 Found checkpoint: {RESUME_CKPT}')
    print(f'   Total checkpoints: {len(normal_ckpts)} normal, {len(exception_ckpts)} exception')
elif exception_ckpts:
    RESUME_CKPT = exception_ckpts[-1]
    print(f'⚠️ Only exception checkpoint found: {RESUME_CKPT}')
else:
    RESUME_CKPT = None
    print('🆕 No checkpoint found — will start from scratch')

In [ ]:
# === TRAINING ===
if RESUME_CKPT:
    print(f'▶️ Resuming from: {RESUME_CKPT}')
    !python train.py \
        --config_filepath configs/train_config_colab.yaml \
        --gpus 0 \
        --resume_from "{RESUME_CKPT}"
else:
    print('▶️ Starting training from scratch...')
    !python train.py \
        --config_filepath configs/train_config_colab.yaml \
        --gpus 0

## 📊 Kiểm tra tiến trình

Xem trên WandB dashboard: https://wandb.ai — project "ArtDapter"

In [ ]:
# Kiểm tra các checkpoint đã lưu
!ls -lh /content/drive/MyDrive/ArtDapter/ckpt/trained/